# Imports

In [ ]:
!pip install unsloth[colab-new] flask pyngrok
!pip install --no-deps xformers==0.0.28.post3

# Hosting

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model.load_adapter("User_Name/Qwen2.5-Coder-7B-Codeforces-Tutor")

FastLanguageModel.for_inference(model)

app = Flask(__name__)

def build_prompt(problem: str, rating: str = "Unknown", topics: str = "Unknown", mode: str = "hint") -> str:
    if mode == "hint":
        return (
            f"Instruction: You are a programming tutor. Give ONE short hint for this problem. "
            f"Do NOT give code or reveal the full solution. Just the key insight in 1-2 sentences.\n"
            f"Difficulty Rating: {rating}\n"
            f"Topics: {topics}\n\n"
            f"Problem:\n{problem}\n\n"
            f"Hint:\n"
        )
    return (
        f"Instruction: You are an expert programmer. Solve the following Codeforces problem in Python.\n"
        f"Difficulty Rating: {rating}\n"
        f"Topics: {topics}\n\n"
        f"Problem:\n{problem}\n\n"
        f"Solution:\n"
    )

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.get_json(force=True)

        if 'inputs' in data:
            prompt = data['inputs']
        else:
            prompt = build_prompt(
                problem=data.get('problem', ''),
                rating=str(data.get('rating', 'Unknown')),
                topics=data.get('topics', 'Unknown'),
                mode=data.get('mode', 'hint')
            )

        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=True,
            temperature=0.3,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
        )

        input_length = inputs["input_ids"].shape[1]
        generated_ids = outputs[0][input_length:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        return jsonify([{"generated_text": response_text}])

    except Exception as e:
        return jsonify({"error": str(e)}), 500

NGROK_TOKEN = "your_ngrok_token_here"
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000).public_url
print(f"Server running at: {public_url}/generate")

app.run(port=5000)